In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LightGCNEncoder(nn.Module):
    def __init__(self, num_users, num_items, embedding_dim=64, num_layers=3):
        super(LightGCNEncoder, self).__init__()
        self.num_users = num_users
        self.num_items = num_items
        self.embedding_dim = embedding_dim
        self.num_layers = num_layers

        # initial embeddings
        self.user_embedding = nn.Embedding(num_users, embedding_dim)
        self.item_embedding = nn.Embedding(num_items, embedding_dim)
        nn.init.normal_(self.user_embedding.weight, std=0.1)
        nn.init.normal_(self.item_embedding.weight, std=0.1)

    def forward(self, norm_adj):
        """
        inputs:
            norm_adj = normalized adjacency matrix
        outputs:
            final_user_embeddings
            final_item_embeddings
        """
        # layer 0 embddings
        users_emb = self.user_embedding.weight
        items_emb = self.item_embedding.weight
        
        # concat of user and items embeddings to make complete matrix
        all_emb = torch.cat([users_emb, items_emb], dim=0)
        embs = [all_emb]

        # message passing
        for layer in range(self.num_layers):
            all_emb = torch.sparse.mm(norm_adj, all_emb)
            embs.append(all_emb)

        # layer combination
        embs = torch.stack(embs, dim=1)
        final_emb = torch.mean(embs, dim=1)

        # splitting final embeddings into user embeddings and item embeddings
        final_user_embeddings, final_item_embeddings = torch.split(
            final_emb, [self.num_users, self.num_items]
        )

        return final_user_embeddings, final_item_embeddings

In [2]:
class BehaviorPrototypeLearning(nn.Module):
    """
    Differentiable behavior prototype learning module.

    Learns separate behavior prototypes for users and items,
    computes soft prototype assignments, and builds
    prototype-aware representations.
    """

    def __init__(
        self,
        embedding_dim=64,
        num_prototypes=5,
        temperature=1.0
    ):
        super(BehaviorPrototypeLearning, self).__init__()
        self.embedding_dim = embedding_dim
        self.num_prototypes = num_prototypes
        self.temperature = temperature
        # Differentiable Behavior Prototype Learning
        self.user_prototypes = nn.Parameter(
            torch.randn(num_prototypes, embedding_dim)
        )
        self.item_prototypes = nn.Parameter(
            torch.randn(num_prototypes, embedding_dim)
        )
        # Prototype Representation  (Wp​ in pape)
        self.user_projection = nn.Linear(
            embedding_dim,
            embedding_dim
        )
        self.item_projection = nn.Linear(
            embedding_dim,
            embedding_dim
        )

    def forward_user(self, embeddings):
        """Compute user prototype assignments and representations."""

        # Distance between users and prototypes
        distances = torch.sum(
            (
                embeddings.unsqueeze(1)
                - self.user_prototypes.unsqueeze(0)
            ) ** 2,
            dim=2
        )
        # Soft prototype assignment
        assignments = F.softmax(
            -distances / self.temperature,
            dim=1
        )
        # Prototype Representation
        prototype_representation = torch.matmul(
            assignments,
            self.user_prototypes
        )
        prototype_representation = self.user_projection(
            prototype_representation
        )
        prototype_representation = torch.sigmoid(
            prototype_representation
        )
        return assignments, distances, prototype_representation

    def forward_item(self, embeddings):
        """Compute item prototype assignments and representations."""

        # Distance between items and prototypes
        distances = torch.sum(
            (
                embeddings.unsqueeze(1)
                - self.item_prototypes.unsqueeze(0)
            ) ** 2,
            dim=2
        )
        # Soft prototype assignment
        assignments = F.softmax(
            -distances / self.temperature,
            dim=1
        )
        # Prototype Representation
        prototype_representation = torch.matmul(
            assignments,
            self.item_prototypes
        )
        prototype_representation = self.item_projection(
            prototype_representation
        )
        prototype_representation = torch.sigmoid(
            prototype_representation
        )

        return assignments, distances, prototype_representation

    def user_prototype_loss(
        self,
        embeddings,
        assignments,
        distances
    ):
        """Compute the user prototype learning loss."""

        loss = torch.sum(
            assignments * distances
        )
        return loss
    
    def item_prototype_loss(
        self,
        embeddings,
        assignments,
        distances
    ):
        """Compute the item prototype learning loss."""
        loss = torch.sum(
            assignments * distances
        )
        return loss

In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class PrototypeGraphRefinement(nn.Module):
    def __init__(self, top_m=10, initial_beta=0.1):
        super(PrototypeGraphRefinement, self).__init__()
        self.top_m = top_m
        
        self.beta = nn.Parameter(torch.tensor(initial_beta, dtype=torch.float32))

    def _get_topk_affinity(self, assignments):
        affinity = torch.matmul(assignments, assignments.T)
        
        num_nodes = assignments.size(0)
        k = min(self.top_m, num_nodes)
        
        vals, indices = torch.topk(affinity, k=k, dim=-1)
        
        mask = torch.zeros_like(affinity)
        mask.scatter_(dim=-1, index=indices, src=torch.ones_like(vals))
        
        sparse_affinity = affinity * mask
        return sparse_affinity

    def forward(self, user_assignments, item_assignments, orig_adj):
        num_users = user_assignments.size(0)
        num_items = item_assignments.size(0)
        total_nodes = num_users + num_items
        
        user_proto_adj = self._get_topk_affinity(user_assignments)
        item_proto_adj = self._get_topk_affinity(item_assignments)
        
        A_proto = torch.zeros(
            (total_nodes, total_nodes), 
            device=user_assignments.device, 
            dtype=user_assignments.dtype
        )
        A_proto[:num_users, :num_users] = user_proto_adj
        A_proto[num_users:, num_users:] = item_proto_adj
        
        if orig_adj.is_sparse:
            orig_adj_dense = orig_adj.to_dense()
        else:
            orig_adj_dense = orig_adj
            
        refined_adj = orig_adj_dense + self.beta * A_proto
        
        return refined_adj, A_proto

In [4]:
import torch
import torch.nn as nn

class AdaptiveGraphMessagePassing(nn.Module):
  
    def __init__(
        self,
        embedding_dim=64,
        num_layers=3,
        num_prototypes=5
    ):
        super(AdaptiveGraphMessagePassing, self).__init__()

        self.embedding_dim = embedding_dim
        self.num_layers = num_layers
        self.num_prototypes = num_prototypes

        # Learnable edge propagation weight
        self.edge_projection = nn.Linear(
            2 * num_prototypes,
            1
        )

    def _compute_propagation_coefficients(
        self,
        assignments
    ):
        num_nodes = assignments.size(0)

        # Build source and target prototype assignments
        source = assignments.unsqueeze(1).expand(
            num_nodes,
            num_nodes,
            -1
        )

        target = assignments.unsqueeze(0).expand(
            num_nodes,
            num_nodes,
            -1
        )

        # Combine assignments for each node pair
        pair_assignments = torch.cat(
            [source, target],
            dim=-1
        )

        # Compute edge propagation coefficients
        alpha = self.edge_projection(
            pair_assignments
        ).squeeze(-1)

        # Map coefficients to [0, 1]
        alpha = torch.sigmoid(alpha)

        return alpha

    def forward(
        self,
        user_embeddings,
        item_embeddings,
        user_assignments,
        item_assignments,
        refined_adj
    ):
        """Perform adaptive message propagation."""

        # Combine user and item embeddings
        all_embeddings = torch.cat(
            [user_embeddings, item_embeddings],
            dim=0
        )

        # Combine prototype assignments
        all_assignments = torch.cat(
            [user_assignments, item_assignments],
            dim=0
        )

        # Compute adaptive propagation weights
        alpha = self._compute_propagation_coefficients(
            all_assignments
        )

        # Keep weights only on existing graph edges
        alpha = alpha * (refined_adj != 0).to(alpha.dtype)

        # NOTE: Degree is computed from the weights of the refined graph
        degrees = torch.sum(
            refined_adj,
            dim=1
        )

        # Avoid division by zero
        degrees = torch.clamp(
            degrees,
            min=1.0
        )

        # Keep embeddings from each propagation depth
        embeddings = [all_embeddings]

        # Propagate messages through the refined graph
        for _ in range(self.num_layers):

            # Compute degree normalization
            normalization = torch.sqrt(
                degrees.unsqueeze(1)
                * degrees.unsqueeze(0)
            )

            # Apply adaptive edge weights and normalization
            normalized_weights = (
                refined_adj * alpha
            ) / normalization

            # Aggregate neighbor messages
            all_embeddings = torch.matmul(
                normalized_weights,
                all_embeddings
            )

            embeddings.append(all_embeddings)

        # Aggregate embeddings from all depths
        embeddings = torch.stack(
            embeddings,
            dim=1
        )

        final_embeddings = torch.mean(
            embeddings,
            dim=1
        )

        return final_embeddings, alpha

In [5]:
import torch
import torch.nn as nn


class LearnableStatisticalEncoder(nn.Module):
    """
    Learnable statistical encoder.

    Maps user and item statistical features into the latent
    embedding space using a two-layer MLP.
    """

    def __init__(
        self,
        input_dim,
        hidden_dim,
        embedding_dim
    ):
        super(LearnableStatisticalEncoder, self).__init__()

        self.user_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim)
        )

        self.item_encoder = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, embedding_dim)
        )

    def forward_user(self, statistical_features):
        """Encode user statistical features."""

        return self.user_encoder(statistical_features)

    def forward_item(self, statistical_features):
        """Encode item statistical features."""

        return self.item_encoder(statistical_features)